# Data Loader

Este notebook carga los cuatro datasets crudos del proyecto:
- Centros Poblados (shapefile)
- Distritos del Perú (shapefile)
- Producción de emergencias por IPRESS (CSVs 2022–2025)
- Establecimientos de salud IPRESS (CSV)

El objetivo es explorar la estructura de cada dataset antes de limpiarlos.



In [1]:
# Librerías principales para el pipeline de datos
import pandas as pd        # manejo de tablas (DataFrames)
import geopandas as gpd    # manejo de datos geoespaciales (GeoDataFrames + shapefiles)
from pathlib import Path   # rutas de archivos compatibles con cualquier sistema operativo


In [2]:
# Definimos la ruta raíz del proyecto
# .parents[1] sube dos niveles desde src/ hasta la carpeta raíz del proyecto
ROOT = Path("..").resolve()

# Rutas a las carpetas de datos crudos
RAW_DIR = ROOT / "data" / "raw"

# Verificamos que las rutas existen antes de continuar
print("Raíz del proyecto:", ROOT)
print("Carpeta raw existe:", RAW_DIR.exists())


Raíz del proyecto: C:\Users\esarmiento\Documents\GitHub\emergency_access_peru
Carpeta raw existe: True


## 1. Establecimientos de salud (IPRESS)

Dataset con todos los establecimientos de salud registrados en el MINSA.
Contiene coordenadas, categoría, distrito y estado de cada establecimiento.


In [4]:
# Cargamos el CSV de establecimientos IPRESS
# encoding='latin1' porque el archivo usa caracteres especiales del español (tildes, ñ)
df_ipress = pd.read_csv(RAW_DIR / "ipress" / "IPRESS.csv", encoding="latin1")

# Vemos dimensiones: cuántas filas (establecimientos) y columnas (atributos)
print("Dimensiones:", df_ipress.shape)

# Vemos los nombres de todas las columnas
print("\nColumnas:")
print(df_ipress.columns.tolist())


Dimensiones: (20819, 33)

Columnas:
['Institución', 'Código Único', 'Nombre del establecimiento', 'Clasificación', 'Tipo', 'Departamento', 'Provincia', 'Distrito', 'UBIGEO', 'Dirección', 'Código DISA', 'Código Red', 'Código Microrred', 'DISA', 'Red', 'Microrred', 'Código UE', 'Unidad Ejecutora', 'Categoria', 'Teléfono', 'Tipo Doc.Categorización', 'Nro.Doc.Categorización', 'Horario', 'Inicio de Actividad', 'Director Médico y/o Responsable de la Atención de Salud', 'Estado', 'Situación', 'Condición', 'Inspección', 'NORTE', 'ESTE', 'COTA', 'CAMAS']


In [5]:
# Primeras filas para entender la estructura del dataset
print(df_ipress[["Código Único", "Nombre del establecimiento", "Categoria", 
                  "Departamento", "Distrito", "UBIGEO", 
                  "NORTE", "ESTE", "Estado"]].head(5).to_string())

# Cuántos valores nulos tienen las coordenadas (NORTE = longitud, ESTE = latitud)
print("\nNulos en NORTE:", df_ipress["NORTE"].isna().sum(), "de", len(df_ipress))
print("Nulos en ESTE: ", df_ipress["ESTE"].isna().sum(), "de", len(df_ipress))

# Valores únicos de Estado (para saber si hay establecimientos inactivos)
print("\nValores de 'Estado':")
print(df_ipress["Estado"].value_counts())


   Código Único Nombre del establecimiento      Categoria Departamento           Distrito  UBIGEO      NORTE      ESTE    Estado
0         16618                 SONRIE MAS            I-1         LIMA  SANTIAGO DE SURCO  150140        NaN       NaN  ACTIVADO
1          7050                     AMBATO            I-1    CAJAMARCA         SANTA CRUZ   60611 -78.858380 -6.133523  ACTIVADO
2            99  SANTA ISABEL DE YUMBATURO            I-1       LORETO           PARINARI  160302 -74.258139 -4.581509  ACTIVADO
3         19555               DENTOCAPLINA  Sin Categoría        TACNA              TACNA  230101        NaN       NaN  ACTIVADO
4         18792  MEDICO DE FAMILIA MANTARA            I-2        JUNIN              TARMA  120701        NaN       NaN  ACTIVADO

Nulos en NORTE: 12863 de 20819
Nulos en ESTE:  12863 de 20819

Valores de 'Estado':
Estado
ACTIVADO    20819
Name: count, dtype: int64


In [6]:
# Distribución por categoría (niveles de atención: I-1 a III-2)
print("Categorías de establecimientos:")
print(df_ipress["Categoria"].value_counts())

# Cuántos establecimientos tienen coordenadas válidas (disponibles para análisis espacial)
con_coords = df_ipress["NORTE"].notna().sum()
print(f"\nEstablecimientos con coordenadas: {con_coords:,} ({con_coords/len(df_ipress)*100:.1f}%)")
print(f"Establecimientos sin coordenadas: {len(df_ipress)-con_coords:,} ({(len(df_ipress)-con_coords)/len(df_ipress)*100:.1f}%)")

# Rango de coordenadas para verificar que están en territorio peruano
print("\nRango de NORTE (longitud):", df_ipress["NORTE"].min(), "a", df_ipress["NORTE"].max())
print("Rango de ESTE  (latitud): ", df_ipress["ESTE"].min(), "a", df_ipress["ESTE"].max())


Categorías de establecimientos:
Categoria
I-1              7260
Sin Categoría    5815
I-2              4128
I-3              2643
I-4               440
II-1              265
II-E              125
II-2               88
III-1              35
III-2              13
III-E               7
Name: count, dtype: int64

Establecimientos con coordenadas: 7,956 (38.2%)
Establecimientos sin coordenadas: 12,863 (61.8%)

Rango de NORTE (longitud): -81.31063225 a 0.0
Rango de ESTE  (latitud):  -18.33541629 a 0.0


In [7]:
# Coordenadas (0.0, 0.0) son inválidas — no corresponden a territorio peruano
coords_cero = df_ipress[(df_ipress["NORTE"] == 0.0) | (df_ipress["ESTE"] == 0.0)]
print(f"Registros con coordenada igual a 0.0: {len(coords_cero)}")

# Resumen de coordenadas disponibles para el análisis espacial
print(f"\nResumen de coordenadas en IPRESS:")
print(f"  Total establecimientos          : {len(df_ipress):>6,}")
print(f"  Sin coordenadas (NaN)           : {df_ipress['NORTE'].isna().sum():>6,}")
print(f"  Con coordenada 0.0 (inválida)   : {len(coords_cero):>6,}")
validas = df_ipress["NORTE"].notna() & (df_ipress["NORTE"] != 0.0) & (df_ipress["ESTE"] != 0.0)
print(f"  Con coordenadas válidas         : {validas.sum():>6,}")


Registros con coordenada igual a 0.0: 3

Resumen de coordenadas en IPRESS:
  Total establecimientos          : 20,819
  Sin coordenadas (NaN)           : 12,863
  Con coordenada 0.0 (inválida)   :      3
  Con coordenadas válidas         :  7,953


### Hallazgos — IPRESS

| Aspecto | Detalle |
|---|---|
| Total establecimientos | 20,819 |
| Con coordenadas válidas | 7,953 (38.2%) |
| Sin coordenadas (NaN) | 12,863 (61.8%) |
| Coordenadas inválidas (0,0) | 3 |
| Estado | 100% ACTIVADO |

**Decisiones de limpieza:**
- Se eliminarán los 3 registros con coordenadas `(0.0, 0.0)`
- Los 12,863 sin coordenadas se conservan para análisis por UBIGEO, pero quedan excluidos del análisis espacial
- `NORTE` = longitud (X), `ESTE` = latitud (Y) — nomenclatura invertida respecto a la convención estándar


## 2. Producción de emergencias por IPRESS (2022–2025)

Dataset con la producción asistencial en emergencias reportada por cada establecimiento,
desagregada por mes, sexo y grupo de edad.
Se tienen 4 archivos anuales que se cargarán y concatenarán en un solo DataFrame.


In [8]:
# Lista con las rutas de cada archivo anual
archivos_emergencias = [
    RAW_DIR / "emergencias" / "ConsultaC1_2022_v24.csv",
    RAW_DIR / "emergencias" / "ConsultaC1_2023_v24.csv",
    RAW_DIR / "emergencias" / "ConsultaC1_2024_v22.csv",
    RAW_DIR / "emergencias" / "ConsultaC1_2025_v20.csv",
]

# Cargamos cada archivo con sep=";" porque usan punto y coma como separador
# y encoding="latin1" por los caracteres especiales del español
dfs = []
for ruta in archivos_emergencias:
    df_temp = pd.read_csv(ruta, sep=";", encoding="latin1")
    print(f"{ruta.name}: {df_temp.shape[0]:,} filas")
    dfs.append(df_temp)

# Unimos todos los años en un solo DataFrame
df_emergencias = pd.concat(dfs, ignore_index=True)
print(f"\nTotal combinado: {df_emergencias.shape[0]:,} filas, {df_emergencias.shape[1]} columnas")


ConsultaC1_2022_v24.csv: 226,189 filas
ConsultaC1_2023_v24.csv: 227,896 filas
ConsultaC1_2024_v22.csv: 250,000 filas
ConsultaC1_2025_v20.csv: 342,753 filas

Total combinado: 1,046,838 filas, 14 columnas


In [9]:
# Columnas y primeras filas para entender qué contiene el dataset
print("Columnas:", df_emergencias.columns.tolist())
print()
print(df_emergencias.head(3).to_string())


Columnas: ['ANHO', 'MES', 'UBIGEO', 'DEPARTAMENTO', 'PROVINCIA', 'DISTRITO', 'SECTOR', 'CATEGORIA', 'CO_IPRESS', 'RAZON_SOC', 'SEXO', 'EDAD', 'NRO_TOTAL_ATENCIONES', 'NRO_TOTAL_ATENDIDOS']

   ANHO  MES  UBIGEO DEPARTAMENTO PROVINCIA    DISTRITO                                   SECTOR CATEGORIA  CO_IPRESS                                     RAZON_SOC     SEXO     EDAD NRO_TOTAL_ATENCIONES NRO_TOTAL_ATENDIDOS
0  2022    5   40101     AREQUIPA  AREQUIPA    AREQUIPA                                  PRIVADO      II-E      11995                           CLINICA ALMONTE SAC  NE_0001  NE_0001              NE_0001             NE_0001
1  2022    5   20101       ANCASH    HUARAZ      HUARAZ  SANIDAD DE LA POLICIA NACIONAL DEL PERU       I-3      10205                        POLICLINICO PNP HUARAZ  NE_0002  NE_0002              NE_0002             NE_0002
2  2022    5  150122         LIMA      LIMA  MIRAFLORES      SANIDAD DE LA FUERZA AEREA DEL PERU     III-1      10751  HOSPITAL CENTRAL DE LA

In [10]:
# Años disponibles en el dataset combinado
print("Años disponibles:", sorted(df_emergencias["ANHO"].unique()))

# Rango de meses por año (para detectar si algún año está incompleto)
print("\nMeses por año:")
print(df_emergencias.groupby("ANHO")["MES"].nunique())

# Valores nulos por columna
print("\nNulos por columna:")
print(df_emergencias.isna().sum())

# Cuántos establecimientos únicos reportan emergencias
print(f"\nEstablecimientos únicos (CO_IPRESS): {df_emergencias['CO_IPRESS'].nunique():,}")

# Cuántos distritos únicos aparecen
print(f"Distritos únicos (UBIGEO): {df_emergencias['UBIGEO'].nunique():,}")


Años disponibles: [np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]

Meses por año:
ANHO
2022    12
2023    12
2024    12
2025    12
Name: MES, dtype: int64

Nulos por columna:
ANHO                    0
MES                     0
UBIGEO                  0
DEPARTAMENTO            0
PROVINCIA               0
DISTRITO                0
SECTOR                  0
CATEGORIA               0
CO_IPRESS               0
RAZON_SOC               0
SEXO                    0
EDAD                    0
NRO_TOTAL_ATENCIONES    0
NRO_TOTAL_ATENDIDOS     0
dtype: int64

Establecimientos únicos (CO_IPRESS): 5,373
Distritos únicos (UBIGEO): 1,190
